In [0]:
CATALOG = "workspace"   # check the Catalog sidebar; change this if yours is named differently

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.landing")

LAND = f"/Volumes/{CATALOG}/bronze/landing"
print("Landing path:", LAND)

Landing path: /Volumes/workspace/bronze/landing


In [0]:
display(dbutils.fs.ls(LAND))

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-23.csv,orders_2026-09-23.csv,32421,1793025412000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-24.csv,orders_2026-09-24.csv,32157,1793025411000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-25.csv,orders_2026-09-25.csv,32392,1793025412000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-26.csv,orders_2026-09-26.csv,32440,1793025412000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-27.csv,orders_2026-09-27.csv,32270,1793025412000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-28.csv,orders_2026-09-28.csv,32292,1793025412000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-29.csv,orders_2026-09-29.csv,32283,1793025411000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-09-30.csv,orders_2026-09-30.csv,32183,1793025411000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-10-01.csv,orders_2026-10-01.csv,32285,1793025412000
dbfs:/Volumes/workspace/bronze/landing/orders_2026-10-02.csv,orders_2026-10-02.csv,32324,1793025411000


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Everything stays as text in bronze, exactly as received, plus lineage columns
orders_raw = (spark.read.option("header", True)
              .csv(f"{LAND}/orders_*.csv")
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_ingested_at", F.current_timestamp()))

(orders_raw.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.orders"))

products_raw = (spark.read.option("header", True).csv(f"{LAND}/products.csv"))
(products_raw.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.products"))

print("bronze.orders rows:  ", spark.table(f"{CATALOG}.bronze.orders").count())
print("bronze.products rows:", spark.table(f"{CATALOG}.bronze.products").count())

bronze.orders rows:   15658
bronze.products rows: 20


In [0]:
raw = spark.table(f"{CATALOG}.bronze.orders")

# 1) Safe type conversion: try_cast gives NULL instead of failing on bad values
checked = (raw
    .withColumn("t_date", F.expr("try_cast(order_date AS DATE)"))
    .withColumn("t_cust", F.expr("try_cast(customer_id AS INT)"))
    .withColumn("t_prod", F.expr("try_cast(product_id AS INT)"))
    .withColumn("t_qty",  F.expr("try_cast(quantity AS INT)"))
    .withColumn("reject_reason",
        F.when(F.trim("order_id").isNull() | (F.trim("order_id") == ""), "missing order_id")
         .when(F.col("t_date").isNull(), "invalid order_date")
         .when(F.col("t_cust").isNull(), "invalid customer_id")
         .when(F.col("t_prod").isNull(), "invalid product_id")
         .when(F.col("t_qty").isNull() | (F.col("t_qty") < 1) | (F.col("t_qty") > 5),
               "quantity out of range")))

# 2) Bad rows go to a quarantine table, with the ORIGINAL values and the reason
quarantine = (checked.filter("reject_reason IS NOT NULL")
              .select(*raw.columns, "reject_reason"))
(quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.orders_quarantine"))

# 3) Good rows get cleaned
good = (checked.filter("reject_reason IS NULL")
    .select(
        F.trim("order_id").alias("order_id"),
        F.col("t_date").alias("order_date"),
        F.col("t_cust").alias("customer_id"),
        F.trim("customer_name").alias("customer_name"),
        F.when(F.trim("city").isNull() | (F.trim("city") == ""), "Unknown")
         .otherwise(F.initcap(F.trim("city"))).alias("city"),
        F.col("t_prod").alias("product_id"),
        F.col("t_qty").alias("quantity"),
        F.when(F.upper(F.trim("payment_type")) == "UPI", "UPI")
         .when(F.upper(F.trim("payment_type")) == "CARD", "Card")
         .when(F.upper(F.trim("payment_type")) == "COD", "COD")
         .when(F.upper(F.trim("payment_type")) == "NETBANKING", "NetBanking")
         .otherwise("Other").alias("payment_type"),
        "_source_file"))

# 4) Deduplicate: keep one row per order_id
w = Window.partitionBy("order_id").orderBy("_source_file")
silver = (good.withColumn("rn", F.row_number().over(w))
              .filter("rn = 1").drop("rn", "_source_file"))

(silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.orders"))

# 5) Products
products = (spark.table(f"{CATALOG}.bronze.products")
    .select(F.expr("try_cast(product_id AS INT)").alias("product_id"),
            F.trim("product_name").alias("product_name"),
            F.trim("category").alias("category"),
            F.expr("try_cast(unit_price AS DECIMAL(10,2))").alias("unit_price"))
    .filter("product_id IS NOT NULL"))
(products.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.products"))

print("good rows before dedup:", good.count())
print("silver.orders rows:    ", spark.table(f"{CATALOG}.silver.orders").count())
print("quarantined rows:      ", spark.table(f"{CATALOG}.silver.orders_quarantine").count())

good rows before dedup: 15655
silver.orders rows:     15500
quarantined rows:       3


In [0]:
display(spark.table(f"{CATALOG}.silver.orders_quarantine"))

order_id,order_date,customer_id,customer_name,city,product_id,quantity,payment_type,_source_file,_ingested_at,reject_reason
null,2026-10-20,1001,No Id,Mumbai,1,2,UPI,dbfs:/Volumes/workspace/bronze/landing/orders_bad_sample.csv,2026-10-26T14:39:59.725Z,missing order_id
X-1,not-a-date,1002,Bad Date,Delhi,2,1,Card,dbfs:/Volumes/workspace/bronze/landing/orders_bad_sample.csv,2026-10-26T14:39:59.725Z,invalid order_date
X-2,2026-10-20,1003,Big Qty,Chennai,3,99,COD,dbfs:/Volumes/workspace/bronze/landing/orders_bad_sample.csv,2026-10-26T14:39:59.725Z,quantity out of range


In [0]:
s = spark.table(f"{CATALOG}.silver.orders")
p = spark.table(f"{CATALOG}.silver.products")

checks = {
  "no null order_id":        s.filter("order_id IS NULL").count(),
  "no duplicate order_id":   s.count() - s.select("order_id").distinct().count(),
  "quantity between 1 and 5": s.filter("quantity NOT BETWEEN 1 AND 5").count(),
  "valid payment_type":      s.filter("payment_type = 'Other'").count(),
  "every product exists":    s.join(p, "product_id", "left_anti").count(),
}
for name, bad in checks.items():
    print(("PASS " if bad == 0 else "FAIL ") + name, "" if bad == 0 else f"({bad} bad rows)")
assert all(v == 0 for v in checks.values()), "Silver quality checks failed"

PASS no null order_id 
PASS no duplicate order_id 
PASS quantity between 1 and 5 
PASS valid payment_type 
PASS every product exists 


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.orders"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-26T14:40:52.000Z,70588434911431,gawasakash70@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""e3da8734-018f-432f-953c-2be40870cfa8"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2700595c-3e81-4351-83b2-6d1c71afbb11"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dee2b16b-70e1-49ac-9fd6-02e07244fd26""}, statsOnLoad -> true)",null,List(1624531293899055),3faa157d-fada-443e-9b2d-f85a888b2dad,1026-142948-zq9vzso8-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 15500, numOutputBytes -> 136226)",null,Databricks-Runtime/19.11.x-photon-scala2.13


In [0]:

display(spark.sql(f"SELECT count(*) AS rows_now FROM {CATALOG}.silver.orders"))
display(spark.sql(f"SELECT count(*) AS rows_v0  FROM {CATALOG}.silver.orders VERSION AS OF 0"))


rows_now
15500


rows_v0
15500
